# Batch inference with the TwitterCLIP media embeddings model

In this notebook we (locally or on DataFlow):
1. Read some data from BQ 
2. Fetch the media from blobstore/mynahbird.
3. Compute the TwitterCLIP media embedding
4. Write the output to BQ.

# Running this notebook

To run this notebook, run `nb create --user <role>` on your laptop.
`<role>` needs to be an accocunt that has permissions to run jobs under the `compute_project` defined in the 
"Configure some parameters for the pipeline" section below, and to write output/create tables under the `storage_project`.

Once the nb instance starts, either upload the notebook through the nb UI. Or run `bootstrap_source` in a terminal in the nb instance and then navigate through the nb UI file browser to this file in source
(`media-understanding/clip/src/main/python/twitter/clip/notebooks/media_embeddings_dataflow.ipynb`).
 

# Setup 

In [ ]:
%pants_load media-understanding/clip/src/main/python/twitter/clip/notebooks:media_embeddings_batch_inference


In [ ]:
import getpass
import os
import subprocess
from datetime import datetime, timedelta

import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
from com.twitter.media_understanding.common import batch_inference
from twitter.ml.common.apache_beam import PipelineOptions as TwitterPipelineOptions


# Configure some parameters for the pipeline

In [ ]:
# Replace these with your own BQ storage (for output table) and compute project

storage_project = "example-storage-project"  
compute_project = "example-media-project"

ts = int(datetime.timestamp(datetime.now()))
output_table_name = f"clip_dataflow_test_{ts}"

# Temporary location on GCS to store temporay output from BQ read and to save a copy of the model somewhere that dataflow can access it.
temp_location_root = "gs://example-dataflow-bucket/beam_temp"

# This is for the media fetch. The media fetch goes straight to the production CDN origin server (mynahbird), so be careful with the request rate (low thousands of QPS).
# Set this to something that will allow the media foundation backend team to easily identify the traffic and get in touch with you if there are any issues.
mynahbird_client_id = f"media_embeddings_{getpass.getuser()}"

run_locally = True  # True for debugging. Set to false to run on dataflow.



In [ ]:

# Cleanup the tempfiles from previous runs to avoid hitting disk quota 

import getpass
import glob
import shutil

user = getpass.getuser()
from pathlib import Path

tmp_dirs = glob.glob("/tmp/tmp*")

for d in tmp_dirs:
  if Path(d).owner() == user:
    if os.path.isfile(os.path.join(d, "media_embeddings_batch_inference.pex")):
      print(f"Found pex file, removing {d}")
      shutil.rmtree(d)
      continue
    if os.path.isfile(os.path.join(d, "setup.py")):
      print(f"Found setup.py file, removing {d}")
      shutil.rmtree(d)
      continue
      
      
# Set the pipeline parameters

if run_locally:
  beam_options = PipelineOptions(
    runner='DirectRunner',
    project=compute_project
  )
  cpus_per_machine = None
else:
  source_root = os.path.join(os.path.expanduser('~'), "workspace", "source")
  pex_file_path = os.path.join(source_root, "dist", "media_embeddings_batch_inference.pex")
  
  subprocess.check_call(["python", "magicpony/tools/build_and_deploy.py", "build_embedding_video", "media-understanding/clip/src/main/python/twitter/clip/notebooks:media_embeddings_batch_inference", "--local", "--no-run"], cwd=source_root)
  
  
  user = getpass.getuser()

  num_workers = 100
  cpus_per_machine = 8  # Must be available in the n1-standard-X range
  number_of_worker_harness_threads = 2 * cpus_per_machine
      
  
  total_cpus = num_workers * cpus_per_machine
  if total_cpus > 1000:
    print("Warning, with {total_cpus} total cpus, you will end up sending a lot of traffic to mynahbird."
          "Please be mindful of this and co-ordinate with the media platform on-call team so they are aware if there any issues." 
          "You can expect around 1 RPS per cpu (to fetch an image and compute the embedding).")
  
  beam_options = TwitterPipelineOptions.for_dataflow_runner(
    service_account=user,
    project=compute_project,
    job_name=f'{user}-media-embeddings-inference-{ts}',
    machine_type=f'n1-standard-{cpus_per_machine}',
    experiments=["use_runner_v2", "no_use_multiple_sdk_containers"],
    extra_pex=pex_file_path, 
    num_workers=num_workers,
    autoscaling_algorithm="NONE",
    number_of_worker_harness_threads=number_of_worker_harness_threads,
  )


# You can tune `machine_type` and `number_of_worker_harness_threads` in the dataflow options above, along with the number of inter and intra op threads for tensorflow.
# Having `number_of_worker_harness_threads` > 1 will mean each worker can fetch multiple images in parallel. 
# Setting this to approximately 2* the number of cpus in the machine type, with `inter_op_threads` = 1 and `intra_op_threads` = 1 or 2 seems to work reasonably well,
# But it may be worth experimenting with these. It might be possible to tune OMP/MKL parameters too though I've not tried this.
# See https://docs.google.com/document/d/1itFsMl4f7hBBK94fYsOC4X3ygVfPA8co8FxzLp9gIDk/edit for tips
# I've found forcing the resuffle to stop fusing doesn't make too much difference to the overall runtime here. With separating them we also risk sending a bit too much traffic
# to mynahbird unless we add a rate limiter.
# Results from a few different configurations here: <internal-experiment-results>

if run_locally:
  inter_op_threads = None
  intra_op_threads = None
  batch_size = 8
else:
  batch_size = 32 * cpus_per_machine
  inter_op_threads = 1
  intra_op_threads = cpus_per_machine
  



# Input query

- This just samples some random tweets from tweetsource. 
- If you have a specific dataset you'd like to compute the embeddings for, replace with your own query.
- You just need to make sure `media_url` is included as a field. You may need to join with tweetsource for this. See the query below to see how to access the media urls. There may be several per Tweet.

In [ ]:
# Number of images to sample from BQ 
if run_locally:
  N_media = 10
else:
  N_media = 100000


In [ ]:
partition_time_dt = datetime.now() - timedelta(days=1)
min_partition_time_dt = partition_time_dt - timedelta(days=3)
min_partition_time_bq_string = datetime.strftime(min_partition_time_dt, "%Y-%m-%d")

input_query = f"""
SELECT
  media_each.media_id,
  ANY_VALUE(media_each.media_url_https) media_url,
  ANY_VALUE(media_each.media_type) media_type,
  ANY_VALUE(media_each.video_variants) video_variants
FROM
  `example-tweetsource-project.user.unhydrated_flat`
CROSS JOIN
  UNNEST (media) AS media_each
WHERE
  DATE(_PARTITIONTIME) >= "{min_partition_time_bq_string}"
  AND ARRAY_LENGTH(media) > 0
GROUP BY media_id
LIMIT
  {N_media}
"""


# Define output schema

The model returns the image embeddings in the field `image_embeddings`. Here we just change the name to `media_embedding` to be consistent with the other fields, 
and covert it to a list so that it is JSON serializable for the BQ write. You can adapt this to whatever fields you read from your input query; they will get passed through the pipeline.

In [ ]:
class FormatOutputForBQ(beam.DoFn):
  def process(self, element):
    import numpy as np

    try:
      embedding = element.pop('media_embedding')
    except KeyError:
      embedding = element.pop('image_embeddings')
      
    element['media_embedding'] = np.asarray(embedding).tolist()
    
    try:
      del element['video_variants']
    except KeyError:
      pass
    
    del element['media']
    yield element


In [ ]:
output_schema = {
  'fields': [
    {
      'name': 'media_id', 'type': 'INTEGER', 'mode': 'REQUIRED'
    }, 
    {
      'name': 'media_embedding', 'type': 'FLOAT', 'mode': 'REPEATED'
    },
    {
      'name': 'media_url', 'type': 'STRING', 'mode': 'REQUIRED'
    },
    {
      'name': 'media_type', 'type': 'INTEGER', 'mode': 'REQUIRED'
    }]
}


# Copy the model from packer to GCS

- The models are published on packer, where they can be easily accessed by anyone. 
- We download the model and copy it to GCS so it can be accessed from the dataflow workers.
- Set the path to somewhere that your `compute_project` has read access.


In [ ]:
model_root_dir_gcs = os.path.join(temp_location_root, "models")
model_dir_gcs = batch_inference.inference.copy_model_from_packer_to_gcs(
  model_root_dir_gcs,
  packer_role="embeddings-category",
  package_name="CLIP-keras-Twitter-ViT-B-32-256",
)

# Run the pipeline

If running on dataflow, see https://console.cloud.google.com/dataflow for job progress

In [ ]:
# To properly process video, we need the video URL. These were only added to TweetSource in Janurary 2022 (in video_variants).
# The only other way to get them is via Media Info Service, we're working on a way to do this from GCP.
# For now, if it's not available in BQ, we can use the embedding of the thumbnail (which is in the media_url_https field). 


In [ ]:
def video_variants_available_in_bq(element):
  return len(element['video_variants']) > 0

def video_variants_not_available_in_bq(element):
  return not (len(element['video_variants']) > 0)


In [ ]:

model_signature = "predict_from_encoded_images"
output_table = f"{storage_project}:user.{output_table_name}"
counter_dir = os.path.join(temp_location_root, f"{output_table_name}_counters")


def add_counter_sink(pipeline, name):
    return pipeline | f"Count {name}" >> beam.combiners.Count.Globally() | f"Write count {name}" >> beam.io.textio.WriteToText(os.path.join(counter_root, name))

    
with beam.Pipeline(options=beam_options) as p:
  bq_input = (p
              | 'Read from BQ' >> beam.io.ReadFromBigQuery(
                query=input_query,
                project=compute_project,
                gcs_location=os.path.join(temp_location_root, "bq_input"),
                use_standard_sql=True)
             )
  
  batch_inference.pipeline.add_counter_sink(bq_input, "input_rows_from_bq", counter_dir)

  images_and_video_thumbnails = (bq_input 
                                | "Filter video variants not available in bq" >> beam.Filter(video_variants_not_available_in_bq)
                                )
      
  embedded_images_and_video_thumbnails = batch_inference.pipeline.add_image_batch_inference_to_pipeline(
    images_and_video_thumbnails,
    mynahbird_client_id,
    model_dir_gcs,
    model_signature=model_signature,
    batch_size=batch_size,
    inter_op_threads=inter_op_threads,
    intra_op_threads=intra_op_threads,
    counter_dir=counter_dir
  )
    
  full_videos =  (bq_input 
             | "Filter video variants available in bq" >> beam.Filter(video_variants_available_in_bq)
             | "Select video variant" >> beam.ParDo(batch_inference.fetch_media.SelectVideoVariant())
            )

  
  embedded_videos = batch_inference.pipeline.add_video_batch_inference_to_pipeline(
    full_videos,
    mynahbird_client_id,
    model_dir_gcs,
    model_signature=model_signature,
    batch_size=batch_size,
    inter_op_threads=inter_op_threads,
    intra_op_threads=intra_op_threads,
    counter_dir=counter_dir
  )
  
  embedded_media = ((embedded_images_and_video_thumbnails, embedded_videos)
               | beam.Flatten()
               )  
  
  
  batch_inference.pipeline.add_counter_sink(embedded_media, "embedded_media", counter_dir)

  formatted = (embedded_media
            | 'Format output' >> beam.ParDo(FormatOutputForBQ())
          )
  
  
  batch_inference.pipeline.add_counter_sink(formatted, "formatted for bq", counter_dir)
  
  formatted | 'Write to BQ' >> beam.io.WriteToBigQuery(
              table=output_table,
              schema=output_schema,
              write_disposition=beam.io.BigQueryDisposition.WRITE_TRUNCATE,
              create_disposition=beam.io.BigQueryDisposition.CREATE_IF_NEEDED,
              custom_gcs_temp_location=os.path.join(temp_location_root, "bq_output"),
              load_job_project_id=compute_project)

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project=compute_project)

results = client.query(f"select * from `{output_table.replace(':', '.')}`").to_dataframe()

results

In [ ]:
import tensorflow as tf

counter_files = tf.io.gfile.listdir(counter_dir)

In [ ]:
for counter_fn in counter_files:
  counter_path = os.path.join(counter_dir, counter_fn)
  with tf.io.gfile.GFile(counter_path) as f:
    counter_name = counter_fn.replace("-00000-of-00001", "")
    count = f.read().strip()
    print(f"{counter_name}: {count}")


In [ ]:
# Total number of output rows:
total_output_rows = client.query(f"select count(DISTINCT media_id) from `{output_table.replace(':', '.')}`").to_dataframe().iloc[0][0]
print(f"Total number of output rows: {total_output_rows}")